# Customer Spending Analytics

This notebook provides a lightweight exploratory workflow for the synthetic retail dataset used by the PostgreSQL and Streamlit project.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
raw_dir = PROJECT_ROOT / 'data' / 'raw'

customers = pd.read_csv(raw_dir / 'customers.csv', parse_dates=['signup_date'])
products = pd.read_csv(raw_dir / 'products.csv')
stores = pd.read_csv(raw_dir / 'stores.csv')
transactions = pd.read_csv(raw_dir / 'transactions.csv', parse_dates=['transaction_date'])

df = (transactions
      .merge(customers, on='customer_id', how='left')
      .merge(products[['product_id', 'product_name', 'category', 'subcategory', 'cost']], on='product_id', how='left')
      .merge(stores[['store_id', 'store_name', 'region']], on='store_id', how='left'))
df.head()

In [ ]:
summary = pd.DataFrame({
    'metric': ['transactions', 'customers', 'products', 'stores', 'revenue', 'profit'],
    'value': [len(df), df['customer_id'].nunique(), df['product_id'].nunique(), df['store_id'].nunique(), df['revenue'].sum(), df['profit'].sum()]
})
summary

In [ ]:
monthly = df.assign(month=df['transaction_date'].dt.to_period('M').dt.to_timestamp()).groupby('month', as_index=False)['revenue'].sum()
px.line(monthly, x='month', y='revenue', markers=True, title='Monthly Revenue Trend')

In [ ]:
category = df.groupby('category', as_index=False).agg(revenue=('revenue', 'sum'), profit=('profit', 'sum'))
category['profit_margin_percent'] = 100 * category['profit'] / category['revenue']
category.sort_values('revenue', ascending=False)

## Interview talking points

- The data generation logic intentionally models repeat customers, best-selling products, discounts, and holiday seasonality.
- SQL analysis covers joins, aggregations, CTEs, window functions, RFM segmentation, churn risk, cohort retention, and rankings.
- Streamlit turns the analysis into an interactive dashboard with filters for executive and category-level exploration.